# UMUD R96T — foldy dekodowane osobno + kąt tekstury z transformaty Radona (Colab, GPU, bez treningu)

**Po co.** Dziś każda rodzina modeli pęczków (R22: 5 foldów, F71: foldy E4) daje **jedną** mapę — średnią po foldach i odbiciu —
i dopiero z niej fragmenty i lex5. W R89T łączenie *późne* (osobne fragmenty i długości, potem średnia) wygrało z łączeniem map
między rodzinami. Tu to samo wewnątrz rodziny: **każdy fold → własne fragmenty i pomiar → połączenie pomiarów.**

| etap | obrazy | co liczymy |
|---|---|---|
| 1 | test (309) + OSF-35 | fragmenty osobno dla każdego foldu (`r22k0…k4`, `f71k0/k2/k3/k4`) i kontrola `r22avg` / `f71avg` (= dzisiejsze fragmenty); kafle Radona; rozcięgna R12B dla OSF-35 |
| 2 | trening (2046), **poza foldem** | model foldu, który obrazu nie widział: dwa widoki odbicia osobno (`nf`, `fl`) i ich średnia (`avg` = dziś) — to samo pytanie „wcześnie czy późno łączyć” na danych z etykietami; kafle Radona |

**Radon (bez modelu):** w kaflach obrazu (1/6 krótszego boku) wariancja rzutów Radona dla kątów −60…60° (co 2°); tekstura pęczków
daje szczyt przy swoim kierunku. Wybór kafli między rozcięgnami i ocena względem śladów GT — u mnie, po biegu.

**Jak uruchomić:** Colab → GPU L4 → **Uruchom wszystko**, zatwierdź Drive. **Czas:** etap 1 ok. 15 min, etap 2 ok. 45–60 min.
Postęp zapisuje się na Drive; po rozłączeniu uruchom ten sam plik ponownie (policzone obrazy i foldy są pomijane).
**Wynik:** `UMUD_Challenge/outputs/R96T_v1_0_0/R96T_wyniki.zip` — prześlij go w czacie (ok. 30–40 MB).
Potrzebne na Drive (jak w R89T): foldy R22 (`outputs/R22_v1_0_0`), foldy F71 (`outputs/R71T_v1_0_0/T`), cache treningowy
`outputs/R71T_v1_0_0/cache_1024.tar`, dane treningowe, obrazy testu i OSF-35, model rozcięgien `UMUD_R12B/best_dinov2_frozen.pt`.


In [ ]:
# =====================================================================================
# KOMÓRKA 1 — USTAWIENIA (jedyne miejsce, które zmieniasz)
# =====================================================================================
RUN_NAME = 'R96T_v1_0_0'     # ten sam katalog = wznowienie
RUN_TEST = True              # etap 1: test + OSF-35 - foldy osobno, kafle Radona, rozcięgna R12B dla OSF (ok. 15 min)
RUN_TRAIN_OOF = True         # etap 2: trening poza foldem - widoki odbicia osobno, kafle Radona (ok. 45-60 min)
SESSION_HOURS = 8.0
CHECK_IMAGE_SHA = True


In [ ]:
# =====================================================================================
# KOMÓRKA 2 — KOD. NIE EDYTUJ: spakowany kod sprawdzany sumą SHA-256 (GPL-3.0-or-later)
# =====================================================================================
import os, sys, json, time, base64, zlib, hashlib, importlib.util, subprocess, shutil, zipfile
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R96T_FIXTURE_ROOT')          # wyłącznie testy lokalne autora
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'; LOCAL.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'
RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
PACKAGE_SHA = '37ad7963713d64d76525c960dcc3adca8745b29071b73fe68e8dc9d294cfdc87'
PAYLOAD = '...'
raw = zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
    raise RuntimeError('Uszkodzony kod notebooka — wczytaj oryginalny plik ponownie')
SRC = LOCAL / ('r96t_src_' + PACKAGE_SHA[:12]); SRC.mkdir(parents=True, exist_ok=True)
for rel, content in json.loads(raw).items():
    path = SRC / rel
    if not path.resolve().is_relative_to(SRC.resolve()):
        raise ValueError('Invalid source path')
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(content)
for module, package in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('cv2', 'opencv-python-headless'),
                        ('skimage', 'scikit-image'), ('timm', 'timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    raise RuntimeError('Potrzebne torch + torchvision. Wybierz standardowe środowisko GPU Colab.')
sys.path.insert(0, str(SRC))
for name in ['r45_core', 'r45_models', 'r45_decode', 'r71_geom', 'r71_sim', 'r71_train', 'r89_tta', 'r96_fold', 'r96_radon']:
    sys.modules.pop(name, None)
import numpy as np, pandas as pd, torch, torchvision, cv2
import r45_core, r45_models, r71_geom, r71_sim, r71_train, r89_tta, r96_fold, r96_radon
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Colab: Środowisko wykonawcze → Zmień typ → GPU.')
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (testy)'
def log(*a):
    msg = ' '.join(str(x) for x in a); print(msg, flush=True)
    with (RUN / 'LOG.txt').open('a') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')
def cuda_gc():
    import gc; gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
code_dir = RUN / 'code'; code_dir.mkdir(exist_ok=True)
archive = code_dir / f'R96T_source_{PACKAGE_SHA[:12]}.zip'
if not archive.exists():
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for rel, content in json.loads(raw).items():
            z.writestr(rel, content)
r45_core.atomic_json(RUN / 'ENVIRONMENT.json', {'python': sys.version, 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'gpu': GPU, 'cuda': torch.version.cuda, 'package_sha256': PACKAGE_SHA,
    'stage': 'R96T per-fold decoding + Radon texture'})
log('R96T start |', GPU, '| kod', PACKAGE_SHA[:12], '|', RUN)


In [ ]:
# =====================================================================================
# KOMÓRKA 3 — DANE I CHECKPOINTY (kontrola sum SHA-256)
# =====================================================================================
ASSETS = SRC / 'assets'
R22RUN = PROJECT / 'outputs' / 'R22_v1_0_0'; R71RUN = PROJECT / 'outputs' / 'R71T_v1_0_0'
TRAIN_ROOT = PROJECT / 'data' / 'competition_train_v1'
SIDE = int(os.environ.get('R96T_SIDE', 768))                    # wejście R22
CFG = dict(r71_train.F71_CONFIG); CFG.update(json.loads(os.environ.get('R96T_TRAIN_OVERRIDES', '{}')))
F_SIDE = int(CFG['side'])                                       # wejście F71 (1024) = bok cache treningowego = bok kafli Radona
ver_path = RUN / 'VERIFIED.json'
ver = json.loads(ver_path.read_text()) if ver_path.exists() else {}
def verified(p, sha):
    key = f'{p}|{p.stat().st_size}|{int(p.stat().st_mtime)}'
    if ver.get(key) != sha:
        assert r45_core.sha256(p) == sha, f'plik różni się od zapisanej sumy: {p}'
        ver[key] = sha
    return p
tc = json.loads((R22RUN / 'TRAINING_COMPLETE.json').read_text())
R22_FOLD_PATH = {int(d['fold']): verified(R22RUN / f"fold_{d['fold']}" / 'final.pt', d['checkpoint_sha256']) for d in tc['checkpoints']}
assert sorted(R22_FOLD_PATH) == [0, 1, 2, 3, 4], 'Potrzebne 5 foldów R22'
R22_IDS = sorted(R22_FOLD_PATH)
F71_FOLD_PATH = {}
for p in sorted((R71RUN / 'T').glob('fold_*/DONE.json')):
    d = json.loads(p.read_text()); F71_FOLD_PATH[int(d['fold'])] = verified(p.parent / 'final.pt', d['final_sha256'])
e4 = R71RUN / 'E4' / 'E4_SUMMARY.json'
F71_E4 = sorted(json.loads(e4.read_text())['folds']) if e4.exists() else sorted(F71_FOLD_PATH)
assert F71_FOLD_PATH and all(f in F71_FOLD_PATH for f in F71_E4), 'Brak foldów F71 z R71T'
R12B_PATH = verified(DRIVE_ROOT / 'UMUD_R12B' / 'best_dinov2_frozen.pt',
                     os.environ.get('R96T_R12B_SHA', '21ec34d0f5dd7664ecab0ca608f976fdf9d2a161e2af2873c21d513cd5ae19ef'))
pairs, prep = r71_train.list_pairs(TRAIN_ROOT / 'fasc_images', TRAIN_ROOT / 'fasc_masks')
FRAME, fold_info = r71_train.folds_from_r22(pairs, R22RUN / 'OOF.csv')
def load_manifest(csv_path, check_sha):
    m = pd.read_csv(csv_path); m['path'] = [str(PROJECT / p) for p in m.relative_path]
    missing = [p for p in m.path if not Path(p).is_file()]
    assert not missing, f'{len(missing)} brakujących obrazów, np. {missing[:3]}'
    if check_sha:
        for r in m.itertuples():
            verified(Path(r.path), r.image_sha256)
    return m
test = load_manifest(Path(os.environ.get('R96T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')), CHECK_IMAGE_SHA)
osf = load_manifest(Path(os.environ.get('R96T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')), CHECK_IMAGE_SHA)
r45_core.atomic_json(ver_path, ver)
TAGS1 = [f'r22k{k}' for k in R22_IDS] + ['r22avg'] + [f'f71k{k}' for k in F71_E4] + ['f71avg']
log('R22 foldów:', R22_IDS, '| F71 foldów:', sorted(F71_FOLD_PATH), '| F71 w cesze (E4):', F71_E4,
    '| trening:', len(FRAME), '| test:', len(test), '| OSF:', len(osf))


In [ ]:
# =====================================================================================
# KOMÓRKA 4 — ETAP 1: test + OSF-35 — każdy fold dekodowany osobno + kontrola (średnia map), kafle Radona, rozcięgna R12B (OSF)
# =====================================================================================
OUT = RUN / 'test'; OUT.mkdir(exist_ok=True)
def stage1_done(name):
    return (OUT / f'{name}_radon.npz').exists() and (OUT / f'{name}_frags_r22avg_t60.csv.gz').exists()
if RUN_TEST:
    todo = [n for n in ('test', 'osf') if not stage1_done(n)]
    if todo:
        r22 = [r45_models.load_r22(R22_FOLD_PATH[f], DEVICE)[0] for f in R22_IDS]
        f71 = [r71_train.load_f71(F71_FOLD_PATH[f], DEVICE)[0] for f in F71_E4]
        r12b = r45_models.load_r12b(R12B_PATH, DEVICE) if 'osf' in todo else None
    for name, rows in (('test', test), ('osf', osf)):
        if name not in todo:
            log(name, ': już policzony'); continue
        def fn(r, gray, name=name):
            hw = gray.shape
            sq71 = r45_core.letterbox_image(gray, F_SIDE)[0]
            fr = r96_fold.per_fold_frags(r96_fold.view_maps(r22, r45_core.letterbox_image(gray, SIDE)[0], DEVICE), R22_IDS, 'r22', hw)
            fr.update(r96_fold.per_fold_frags(r96_fold.view_maps(f71, sq71, DEVICE), F71_E4, 'f71', hw))
            out = {'frags': fr, 'radon': r96_radon.encode(*r96_radon.radon_tiles(np.rint(sq71 * 255.) / 255., hw))}
            if name == 'osf':
                out['apo'] = {'r12b': r71_geom.apo_pair_record(r45_models.predict_r12b(r12b, r.path, DEVICE))}
            return out
        sig = {'package': PACKAGE_SHA, 'stage': name, 'side_r22': SIDE, 'side_f71': F_SIDE, 'r22': {k: str(p) for k, p in R22_FOLD_PATH.items()},
               'f71': {k: str(F71_FOLD_PATH[k]) for k in F71_E4}, 'r12b': str(R12B_PATH), 'radon': [r96_radon.TILE_FRAC, r96_radon.WORK, r96_radon.PAD]}
        recs = r96_fold.run_images(name, rows[['image_id', 'path']], OUT / '_cache', fn, sig, log=log)
        r96_fold.export_tags(name, recs, OUT, TAGS1); r96_radon.export(name, recs, OUT)
        bad = [r['image_id'] for r in recs if r['status'] != 'OK']
        nfr = {t: sum(len(((r.get('frags') or {}).get(t) or {}).get('0.60') or []) for r in recs) for t in TAGS1}
        log(f'{name} GOTOWY: {len(recs)} obrazów, błędy {len(bad)} {bad[:5]} | fragmenty na obraz:',
            {t: round(v / max(1, len(recs)), 1) for t, v in nfr.items()})
    if todo:
        del r22, f71, r12b; cuda_gc()
    log('ETAP 1 GOTOWY')


In [ ]:
# =====================================================================================
# KOMÓRKA 5 — ETAP 2: trening POZA FOLDEM — widoki odbicia dekodowane osobno + średnia (= dziś), kafle Radona; wznawialne per fold
# =====================================================================================
OUTT = RUN / 'train'; OUTT.mkdir(exist_ok=True)
if RUN_TRAIN_OOF:
    CACHE = LOCAL / f'r96t_cache_{F_SIDE}'
    cidx = r71_train.sync_cache(FRAME, CACHE, R71RUN / f'cache_{F_SIDE}.tar', F_SIDE, CFG, log=log)
    deadline = T_START + SESSION_HOURS * 3600
    for fold in sorted(FRAME.fold.unique()):
        fdir = OUTT / f'fold_{fold}'; flag = fdir / 'DONE.json'
        if flag.exists():
            log(f'fold {fold}: już gotowy'); continue
        if time.time() > deadline:
            log('limit sesji — uruchom ponownie'); break
        held = FRAME[FRAME.fold == fold]; fdir.mkdir(parents=True, exist_ok=True)
        m22 = r45_models.load_r22(R22_FOLD_PATH[int(fold)], DEVICE)[0]
        m71 = r71_train.load_f71(F71_FOLD_PATH[int(fold)], DEVICE)[0] if int(fold) in F71_FOLD_PATH else None
        recs = []; t0 = time.time()
        for k, r in enumerate(held.itertuples(index=False)):
            rec = {'image_id': r.name}
            try:
                meta = json.loads((CACHE / f'{r.name}.json').read_text()); hw = (int(meta['h']), int(meta['w']))
                x = np.asarray(r71_train.load_sample(CACHE, r.name)[0], np.float32)
                sq = r89_tta.CacheSquare(x, hw)
                fr = r96_fold.per_view_frags(r96_fold.view_maps([m22], sq(SIDE), DEVICE)[0], 'r22', hw)
                if m71 is not None:
                    fr.update(r96_fold.per_view_frags(r96_fold.view_maps([m71], sq(F_SIDE), DEVICE)[0], 'f71', hw))
                rec.update(h=hw[0], w=hw[1], status='OK', frags=fr, radon=r96_radon.encode(*r96_radon.radon_tiles(x, hw)))
            except Exception as e:  # noqa: BLE001 - jeden obraz nie zatrzymuje biegu
                rec['status'] = f'ERROR:{type(e).__name__}:{e}'[:300]
            recs.append(rec)
            if (k + 1) % 100 == 0:
                log(f'fold {fold}: {k + 1}/{len(held)} ({time.time() - t0:.0f} s)')
        tags = ['r22nf', 'r22fl', 'r22avg'] + (['f71nf', 'f71fl', 'f71avg'] if m71 is not None else [])
        r96_fold.export_tags(f'train_fold{fold}', recs, fdir, tags); r96_radon.export(f'train_fold{fold}', recs, fdir)
        bad = [r['image_id'] for r in recs if r['status'] != 'OK']
        r45_core.atomic_json(flag, {'fold': int(fold), 'n': int(len(held)), 'errors': len(bad), 'seconds': round(time.time() - t0, 1)})
        del m22, m71; cuda_gc(); log(f'fold {fold} GOTOWY ({len(held)} obrazów, błędy {len(bad)} {bad[:3]})')
    log('ETAP 2: foldy gotowe:', sorted(int(p.parent.name.split('_')[1]) for p in OUTT.glob('fold_*/DONE.json')))


In [ ]:
# =====================================================================================
# KOMÓRKA 6 — JEDEN plik wynikowy do przesłania
# =====================================================================================
ZIP = RUN / 'R96T_wyniki.zip'; tmp = ZIP.with_suffix('.tmp.zip')
with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN.rglob('*')):
        rel = p.relative_to(RUN)
        if p.is_file() and '_cache' not in rel.parts and (p.name.endswith(('_t60.csv.gz', '_images.csv', '_apo.csv.gz', '_radon.npz'))
                                                          or p.name in ('LOG.txt', 'ENVIRONMENT.json', 'DONE.json')):
            z.write(p, str(rel))
tmp.replace(ZIP)
log('GOTOWE. Prześlij w czacie:', ZIP, f'({ZIP.stat().st_size / 1e6:.1f} MB)')
